# DP

In [ ]:
model = MyModel()
model = torch.nn.DataParallel(model, device_ids=[0, 1])  # запустить на GPU 0 и 1
output = model(input) 

# DDP

In [ ]:
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

def setup(rank, world_size):
    # Инициализация группы процессов для DDP (здесь nccl для GPU)
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    dist.init_process_group("nccl", rank=rank, world_size=world_size)

def demo_ddp(rank, world_size):
    setup(rank, world_size)
    # Создаём модель и переносим её на соответствующий GPU
    model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased').to(rank)
    ddp_model = DDP(model, device_ids=[rank])
    # Дальше обычный цикл обучения...
    dist.destroy_process_group()

if __name__ == "__main__":
    world_size = 2  # количество GPU
    torch.multiprocessing.spawn(demo_ddp, args=(world_size,), nprocs=world_size)

# FSDP

In [ ]:
import torch
from torch.distributed import ReduceOp

# локальный тензор (содержимое у каждого ранга разное)
x = torch.tensor([1.0, 2.0]).cuda()

# суммируем по всем рангам и записываем обратно в x
torch.distributed.all_reduce(x, op=ReduceOp.SUM)

# если нужен average:
world_size = torch.distributed.get_world_size()

In [ ]:
local = torch.arange(3).cuda() + torch.distributed.get_rank() * 10  # пример: [0,1,2], [10,11,12], ...
gathered = [torch.zeros_like(local) for _ in range(torch.distributed.get_world_size())]
torch.distributed.all_gather(gathered, local)
# теперь gathered = [local_from_rank0, local_from_rank1, ...] на каждом ранге
full = torch.cat(gathered, dim=0)  # если хотим конкатенировать

In [ ]:
from torch.distributed import ReduceOp

world_size = torch.distributed.get_world_size()
# У каждого ранга список length == world_size; обычно это куски большого тензора
local_list = [torch.ones(4).cuda() * (torch.distributed.get_rank() + i) for i in range(world_size)]
# out - размер одной "шарды"
out = torch.zeros(4).cuda()
torch.distributed.reduce_scatter(out, local_list, op=ReduceOp.SUM)
# out будет суммой соответствующих элементов и отдан именно этому рангу

# Sample 1

In [ ]:
import os
import torch
import torch.distributed as dist
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP

def setup():
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    global_rank = int(os.environ.get("RANK", local_rank))
    world_size = int(os.environ.get("WORLD_SIZE", 1))

    torch.cuda.set_device(local_rank)

    tmp = torch.zeros(1, device=f"cuda:{local_rank}")
    del tmp
    torch.cuda.synchronize()

    if world_size > 1:
        dist.init_process_group(backend="nccl", init_method="env://", rank=global_rank, world_size=world_size)

    return global_rank, world_size, local_rank

def demo_fsdp():

    global_rank, world_size, local_rank = setup()
    device = torch.device(f"cuda:{rank}")

    model_name = "bert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

    # Загружаем и переносим модель на локальный GPU
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

    # Простейшая обёртка FSDP - оборачиваем всю модель
    fsdp_model = FSDP(model)

# Sample 2

In [ ]:
import functools
import torch
import torch.distributed as dist
from torch.distributed.fsdp import (
    FullyShardedDataParallel as FSDP,
    CPUOffload,
    MixedPrecision,
    ShardingStrategy,
    FullStateDictConfig,
    StateDictType,
)
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy
from transformers import AutoModelForCausalLM

def make_fsdp_model(args, local_rank, global_rank, world_size):
    """
    Создаём модель, настраиваем auto-wrap policy для трансформерных блоков и
    оборачиваем в FSDP с опциями sharding/mixed-precision/cpu-offload.
    """
    device = torch.device(f"cuda:{local_rank}")
    model = AutoModelForCausalLM.from_pretrained(args.model_name).to(device)

    # Подбор класса трансформерного блока: берём первый блок модели (универсальная идея для HF-трансформеров)
    # NB: у разных архитектур путь к блокам может отличаться (e.g. model.transformer.h, model.encoder.layer, ...)
    transformer_block_cls = model.transformer.h[0].__class__

    auto_wrap_policy = None
    if transformer_block_cls is not None:
        auto_wrap_policy = functools.partial(
            transformer_auto_wrap_policy,
            transformer_layer_cls={transformer_block_cls},
            min_num_params=args.min_wrap_params,
        )

    mp = MixedPrecision(param_dtype=torch.float16, reduce_dtype=torch.float16, buffer_dtype=torch.float16)
    cpu_offload = CPUOffload(offload_params=args.cpu_offload)

    fsdp_model = FSDP(
        model,
        sharding_strategy=ShardingStrategy.FULL_SHARD,
        auto_wrap_policy=auto_wrap_policy,
        cpu_offload=cpu_offload,
        mixed_precision=mp,
        backward_prefetch="BACKWARD_PRE",  # вариант: "NO_PREFETCH", "BACKWARD_PRE", "BACKWARD_POST"
    )

    return fsdp_model

# Checkpoints

In [ ]:
import torch.distributed.checkpoint as dcp
from torch.distributed.checkpoint.state_dict import get_state_dict
from torch.distributed.fsdp.fully_sharded_data_parallel import StateDictType

def save_fsdp_checkpoint(fsdp_model, optimizer, path):
    # Получаем sharded state dict модели
    model_state = get_state_dict(fsdp_model, StateDictType.SHARDED_STATE_DICT)
    ckpt = {"model": model_state, "optimizer": optimizer.state_dict()}
    # Сохраняем параллельно с каждого ранга в path (DCP создаст файлы per-rank)
    dcp.save(ckpt, path)